In [18]:
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.arima.model import ARIMA

# Setup path
sys.path.insert(0, str(Path.cwd().parent.parent))

# ==========================================
# 1. Feature Engineering & Utility Functions
# ==========================================

def add_rolling_stats(df, col='mid_price', window=20):
    """
    Refined version to ensure stability and direction.
    """
    # 1. Rolling Mean & Std Dev
    rolling_mean = df[col].rolling(window=window).mean()
    # ddof=1 is best for sampling; pandas handles NaNs automatically
    rolling_std = df[col].rolling(window=window).std(ddof=1)
    
    # 2. Variance (Guaranteed >= 0)
    # If this is negative, pandas/numpy has a memory alignment issue.
    # We clip at 0 just to be safe for downstream square-rooting.
    df[f'{col}_rolling_var'] = (rolling_std**2).clip(lower=0)
    
    # 3. Z-Score (Directional)
    # We use .replace to avoid 'infinity' if price is perfectly flat (std=0)
    df[f'{col}_rolling_z_score'] = (df[col] - rolling_mean) / rolling_std.replace(0, np.nan)
    
    return df

def add_mid_prices(df):
    """Calculates and adds mid_price and wall_mid columns."""
    df['mid_price'] = (df['bid_price_1'] + df['ask_price_1']) / 2
    df['wall_mid'] = (df['bid_price_2'] + df['ask_price_2']) / 2
    return df

def calculate_obi(df, levels=1):
    """
    Calculates Order Book Imbalance (OBI) up to the specified number of depth levels.
    """
    bid_cols = [f'bid_volume_{i}' for i in range(1, levels + 1)]
    ask_cols = [f'ask_volume_{i}' for i in range(1, levels + 1)]
    
    total_bid_vol = df[bid_cols].sum(axis=1)
    total_ask_vol = df[ask_cols].sum(axis=1)
    
    imbalance = (total_bid_vol - total_ask_vol) / (total_bid_vol + total_ask_vol)
    return imbalance.fillna(0).replace(0, np.nan)

def get_slope(y):
    """Calculates the linear regression slope for rolling windows."""
    x = np.arange(len(y))
    return np.polyfit(x, y, 1)[0]

def add_rolling_slope(df, col_name, window=50):
    """Calculates and adds the rolling slope of a specified column."""
    df['rolling_slope'] = df[col_name].rolling(window=window).apply(get_slope, raw=True)
    df.loc[df.index[:window], 'rolling_slope'] = np.nan
    return df

# ==========================================
# 2. ARIMA Modeling Functions
# ==========================================

def clean_arima(res, target_index):
    """Extracts fitted values and aligns them with the target index, turning the first value to NaN."""
    data = res.fittedvalues if hasattr(res, 'fittedvalues') else res
    s = pd.Series(data, index=target_index)
    s.iloc[0] = np.nan
    return s

def apply_arima_models(df, i_label):
    """Fits ARIMA models to mid_price and wall_mid, prints summaries, and applies cross-smoothing."""
    model_mid = ARIMA(df['mid_price'].values, order=(0, 1, 1)) 
    model_wall = ARIMA(df['wall_mid'].values, order=(0, 1, 1)) 

    res_mid = model_mid.fit()
    res_wall = model_wall.fit()

    print(f"\n{'='*80}\nDATAFRAME {i_label} - MID PRICE ARIMA(0,1,1)\n{'='*80}")
    print(res_mid.summary())

    print(f"\n{'='*80}\nDATAFRAME {i_label} - WALL PRICE ARIMA(0,1,1)\n{'='*80}")
    print(res_wall.summary())

    # Generate cross-smoothed parameters
    res_wall_reversed = model_wall.smooth(res_mid.params)
    mid_price_reversed = model_mid.smooth(res_wall.params)

    # Attach results to dataframe
    df['ARIMA_mid'] = clean_arima(res_mid, df.index)
    df['ARIMA_wall'] = clean_arima(res_wall, df.index)
    df['ARIMA_wall_rever'] = clean_arima(res_wall_reversed, df.index)
    df['ARIMA_mid_rever'] = clean_arima(mid_price_reversed, df.index)
    
    return df

def spread_to_fill(df):
    df['spread'] = ((df['ARIMA_wall_rever']-df['bid_price_1']+1) + (df['ask_price_1']-df['ARIMA_wall_rever']-1))/2

# ==========================================
# 3. Data Loading & Log Parsing Functions
# ==========================================

def load_trade_history(file_path, symbol='TOMATOES'):
    """Parses the trade history from a JSON log file into a Pandas DataFrame."""
    with open(file_path, 'r') as file:
        content = file.read().lstrip() # Strips any accidental leading characters
        
    log_data = json.loads(content)
    df_fills = pd.DataFrame(log_data.get('tradeHistory', []))
    
    if not df_fills.empty:
        df_fills = df_fills[df_fills['symbol'] == symbol].reset_index(drop=True)
        
    return df_fills


# ==========================================
# Main Execution Pipeline
# ==========================================

# 1. Load Data
price_df_1 = pd.read_csv("data/prices_round_0_day_-2.csv", sep=';')
price_df_2 = pd.read_csv("data/prices_round_0_day_-1.csv", sep=';')

# 2. Filter for TOMATOES
dfs = [
    price_df_2[price_df_2['product'] == 'TOMATOES'].reset_index(drop=True),
    price_df_1[price_df_1['product'] == 'TOMATOES'].reset_index(drop=True)
]

# 3. Process each DataFrame
for i, df in enumerate(dfs):
    # Apply basic pricing logic
    df = add_mid_prices(df)
    
    # Apply Statistical Analysis (New functions used here
    
    # Calculate Imbalances
    df['obi'] = calculate_obi(df, levels=1)
    df['obi3'] = calculate_obi(df, levels=3)
    
    # Determine Signals
    obi_present = df['obi'].notna()
    obi3_present = df['obi3'].notna()
    only_obi3 = obi3_present & ~obi_present
    
    df['signal'] = only_obi3.astype(int)
    df['obi3_s'] = df['obi3'].where(only_obi3)
    df['bid_price_3_s'] = df['bid_price_3'].where(only_obi3)
    df['ask_price_3_s'] = df['ask_price_3'].where(only_obi3)
    
    print(f"\nProcessing DF {i} Signals:")
    print(f"Only obi3: {only_obi3.sum()} | Only obi: {(obi_present & ~obi3_present).sum()} | Both: {(obi_present & obi3_present).sum()} | Neither: {(~obi_present & ~obi3_present).sum()}")
    
    # Apply Models and Slope
    df = apply_arima_models(df, i_label=i)
    df = add_rolling_stats(df, "ARIMA_wall_rever")
    df = add_rolling_stats(df)

    spread_to_fill(df)


# 4. Load JSON Fills Data
df_fills = load_trade_history('27396.log')
print(f"\nFirst 5 fills:\n{df_fills.head(5)}")
print(f"Total fills loaded: {len(df_fills)}")


Processing DF 0 Signals:
Only obi3: 58 | Only obi: 0 | Both: 664 | Neither: 9278

DATAFRAME 0 - MID PRICE ARIMA(0,1,1)
                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                10000
Model:                 ARIMA(0, 1, 1)   Log Likelihood              -15794.552
Date:                Mon, 13 Apr 2026   AIC                          31593.104
Time:                        16:19:11   BIC                          31607.525
Sample:                             0   HQIC                         31597.986
                              - 10000                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ma.L1         -0.5553      0.007    -82.459      0.000      -0.568      -0.542
sigma2     

In [22]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Create a figure with 2 rows
price_graph = make_subplots(
    rows=2, 
    cols=1, 
    specs=[[{"secondary_y": True}],  # Row 1
           [{"secondary_y": True}]]  # Row 2
)

# 1. Merge fills with the order flow data to get 'wall_mid_rever' on the same row as fills
# Note: 'df' here refers to the specific product's order flow dataframe from your loop
df_merged = pd.merge(df_fills, df[['timestamp', 'ARIMA_wall_rever']], on='timestamp', how='left')

# 2. Create the column: show the price if it's within 4 units, otherwise NaN
df_merged['fills_near_wall'] = np.where(
    abs(df_merged['price'] - df_merged['ARIMA_wall_rever']) <= 5, 
    df_merged['price'], 
    np.nan
)

for i, df in enumerate(dfs):
        row = i + 1
        price_graph.add_trace(go.Scatter(x=df['timestamp'], y=df['ARIMA_wall_rever'], mode='lines', name='ARIMA Wall Reversed'), row=row, col=1)

        price_graph.add_trace(go.Scatter(x=df['timestamp'], y=df['mid_price_rolling_var'], name='Variance'), row=row, col=1, secondary_y=True)
        price_graph.add_trace(go.Scatter(x=df['timestamp'], y=df['mid_price_rolling_z_score'], name='Z-Score'), row=row, col=1, secondary_y=True)
        price_graph.add_trace(go.Scatter(x=df['timestamp'], y=df['spread'], name='spread'), row=row, col=1, secondary_y=True)

price_graph.update_layout(height=800, title_text="Tomato Price Comparison")
price_graph.show()

In [23]:
df.head(5)

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,...,ask_price_3_s,ARIMA_mid,ARIMA_wall,ARIMA_wall_rever,ARIMA_mid_rever,ARIMA_wall_rever_rolling_var,ARIMA_wall_rever_rolling_z_score,mid_price_rolling_var,mid_price_rolling_z_score,spread
0,-2,0,TOMATOES,4993,7,4992,17,NaN,NaN,5007,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-2,100,TOMATOES,4998,5,4993,7,4992.0,16.0,5007,...,NaN,4999.996074,4999.999526,4999.996074,4999.999526,NaN,NaN,NaN,NaN,4.5
2,-2,200,TOMATOES,4994,6,4993,20,NaN,NaN,5008,...,NaN,5001.418011,5000.394305,5000.282245,5001.971926,NaN,NaN,NaN,NaN,7.0
3,-2,300,TOMATOES,4995,5,4993,21,NaN,NaN,5008,...,NaN,5001.222102,5000.866112,5000.618636,5001.214843,NaN,NaN,NaN,NaN,6.5
4,-2,400,TOMATOES,4995,8,4993,20,NaN,NaN,5008,...,NaN,5001.344128,5000.970340,5000.786094,5001.436829,NaN,NaN,NaN,NaN,6.5


In [28]:

import numpy as np
import matplotlib.pyplot as plt


price = df['wall_mid'].values

# =========================
# PARAMETERS (TUNE THESE)
# =========================
fast_alpha = 0.4
slow_alpha = 0.05
accel_window = 50
z_thresh = 2.0

# =========================
# DUAL EWMA
# =========================
s_fast = np.zeros_like(price)
s_slow = np.zeros_like(price)

for i in range(len(price)):
    if i == 0:
        s_fast[i] = price[i]
        s_slow[i] = price[i]
    else:
        s_fast[i] = fast_alpha * price[i] + (1-fast_alpha) * s_fast[i-1]
        s_slow[i] = slow_alpha * price[i] + (1-slow_alpha) * s_slow[i-1]

# =========================
# VELOCITY + ACCELERATION
# =========================
velocity = s_fast - s_slow
acceleration = np.diff(velocity, prepend=velocity[0])

# Rolling std for normalization
vel_std = pd.Series(velocity).rolling(accel_window).std().bfill().values
z_accel = acceleration / (vel_std + 1e-6)

# =========================
# CROSSOVER SIGNAL
# =========================
crossover = np.zeros_like(price)

for i in range(1, len(price)):
    if s_fast[i] > s_slow[i] and s_fast[i-1] <= s_slow[i-1]:
        crossover[i] = 1  # bullish cross
    elif s_fast[i] < s_slow[i] and s_fast[i-1] >= s_slow[i-1]:
        crossover[i] = -1  # bearish cross

# =========================
# REGIME DETECTION
# =========================
regime = np.zeros_like(price)

for i in range(len(price)):
    if z_accel[i] > z_thresh:
        regime[i] = 1
    elif z_accel[i] < -z_thresh:
        regime[i] = -1
    else:
        regime[i] = 0

# =========================
# ROLLING CUMULATIVE VELOCITY
# =========================
cum_window = 50  # match your drift horizon (tune this)

rolling_cum_vel = (
    pd.Series(velocity)
    .rolling(cum_window)
    .sum()
    .bfill()
    .values
)

# =========================
# PLOTTING
# =========================
import plotly.graph_objects as go

# =========================
# PRICE + EWMA
# =========================
fig = go.Figure()

fig.add_trace(go.Scatter(y=price, mode='lines', name='Price'))
fig.add_trace(go.Scatter(y=s_fast, mode='lines', name='Fast EWMA'))
fig.add_trace(go.Scatter(y=s_slow, mode='lines', name='Slow EWMA'))

fig.update_layout(title='Price + Dual EWMA')
fig.show()


# =========================
# VELOCITY
# =========================
fig = go.Figure()

fig.add_trace(go.Scatter(y=velocity, mode='lines', name='Velocity'))

fig.update_layout(title='Velocity (fast - slow)')
fig.show()


# =========================
# ACCELERATION (Z-SCORE)
# =========================
fig = go.Figure()

fig.add_trace(go.Scatter(y=z_accel, mode='lines', name='Z-Accel'))
fig.add_trace(go.Scatter(y=[z_thresh]*len(z_accel), mode='lines', name='Upper Threshold'))
fig.add_trace(go.Scatter(y=[-z_thresh]*len(z_accel), mode='lines', name='Lower Threshold'))

fig.update_layout(title='Z-Scored Acceleration')
fig.show()


# =========================
# CROSSOVER SIGNALS
# =========================
fig = go.Figure()

fig.add_trace(go.Scatter(y=price, mode='lines', name='Price'))
fig.add_trace(go.Scatter(x=df.index, y=df['bid_price_3'], mode='markers', name='bid_price_3'))
fig.add_trace(go.Scatter(x=df.index, y=df['ask_price_3'], mode='markers', name='ask_price-3'))

# Bullish
bull_idx = np.where(crossover == 1)[0]
fig.add_trace(go.Scatter(
    x=bull_idx,
    y=price[bull_idx],
    mode='markers',
    name='Bullish Cross',
    marker=dict(symbol='triangle-up', size=8)
))

# Bearish
bear_idx = np.where(crossover == -1)[0]
fig.add_trace(go.Scatter(
    x=bear_idx,
    y=price[bear_idx],
    mode='markers',
    name='Bearish Cross',
    marker=dict(symbol='triangle-down', size=8)
))

fig.update_layout(title='Crossover Signals')
fig.show()


# =========================
# REGIME SIGNALS
# =========================
fig = go.Figure()

fig.add_trace(go.Scatter(y=price, mode='lines', name='Price'))

# Up regime
up_idx = np.where(regime == 1)[0]
fig.add_trace(go.Scatter(
    x=up_idx,
    y=price[up_idx],
    mode='markers',
    name='Up Regime',
    marker=dict(size=6)
))

# Down regime
down_idx = np.where(regime == -1)[0]
fig.add_trace(go.Scatter(
    x=down_idx,
    y=price[down_idx],
    mode='markers',
    name='Down Regime',
    marker=dict(size=6)
))

fig.update_layout(title='Acceleration-Based Regimes')
fig.show()

# =========================
# ROLLING CUMULATIVE VELOCITY
# =========================
fig = go.Figure()

fig.add_trace(go.Scatter(y=rolling_cum_vel, mode='lines', name='Rolling Cum Velocity'))

fig.update_layout(title='Rolling Cumulative Velocity')
fig.show()